# 16 – Phase 5b: a larger backbone (ConvNeXt-Small)

**Question:** is ConvNeXt-Tiny's plateau at OOF ≈ 0.85 a **capacity** limit?

Notebooks 11–15 changed augmentation, resolution and labels without moving it. The remaining errors are disturbed (F1 ≈ 0.5) and the spiral boundaries. A larger network can, in principle, learn subtler cues, e.g. the faint tidal features of disturbed galaxies.

**One change at a time.** Everything identical to `ref_256` (notebook 14), except the backbone:

| | ConvNeXt-Tiny (`ref_256`) | ConvNeXt-Small (`cnxs_256`) |
|---|---|---|
| parameters | 28 M | 50 M |
| depth (blocks per stage) | 3-3-9-3 | 3-3-**27**-3 |
| channels | 96-192-384-768 | 96-192-384-768 (the same) |
| pre-training | ImageNet-22k → 1k | ImageNet-22k → 1k (the same) |
| training cost | 1× (≈ 16 min) | ≈ 2× (≈ 30 min) |

ConvNeXt-Small has the same width as Tiny but three times as many blocks in the third stage. It is "Tiny, deeper", so the comparison really isolates capacity. Learning rate, drop path (0.1), augmentation, epochs and batch are left unchanged on purpose, to keep the change single. A larger model sometimes wants more regularisation; that would be a later step, and only if Small looks promising.

**A second run: the same `ref_256`, with a different seed** (`ref_256_s43`). Notebook 14 estimated the run-to-run noise indirectly, from two *almost* identical recipes. Here we measure it directly, with the same recipe and a different seed, i.e. a different head initialisation, batch order and random augmentations. This gives the decision a real threshold: Small is convincing only if its gain is clearly larger than the difference between two seeds of Tiny.

**Decision rule:** adopt ConvNeXt-Small if its OOF F1 beats `ref_256` by **≳ 0.01 on both validations**, and by clearly more than the seed-to-seed difference. Otherwise Tiny stays, because it is half the cost.

**Time:** ≈ 30 min (Small) + 16 min (Tiny, seed 43) ≈ **50 min**. Run it in `tmux` from the VS Code terminal:
```
tmux new -s phase5b
cd ~/ANN-DL-Challenge/families/elia && source ../../.venv/bin/activate
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 16_phase5b_convnext_small.ipynb
```
**Close the notebook in VS Code before launching** (and do not save it while it runs), otherwise VS Code may overwrite the executed file, as happened with notebook 15.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

import train_lib as T
import common as C

plt.rcParams["figure.dpi"] = 80
SMOKE_TEST = False          # True = tiny run (few images, 1 epoch, fold 0) only to check that the code works
data = T.load_data(smoke_test=SMOKE_TEST)
REF = "ref_256"             # trained in notebook 14
print("device:", T.DEVICE, "| outputs:", T.OUT_DIR)
assert SMOKE_TEST or (T.OUT_DIR / REF / "oof.csv").exists(), "run notebook 14 first (ref_256)"

## 1. The experiments

In [ ]:
EXPERIMENTS = [
    T.reference_config(run_name="cnxs_256", note="reference with ConvNeXt-Small (ImageNet-22k -> 1k)",
                       model="convnext_small.fb_in22k_ft_in1k"),
    T.reference_config(run_name="ref_256_s43", note="reference (ConvNeXt-Tiny), seed 43: run-to-run noise", seed=43),
]
if SMOKE_TEST:
    for cfg in EXPERIMENTS:
        cfg.update(run_name=cfg["run_name"] + "_SMOKE", epochs=1, folds=[0], batch_size=16)
pd.DataFrame(EXPERIMENTS)[["run_name", "model", "seed", "img_size", "epochs", "batch_size", "lr", "drop_path"]]

In [ ]:
results = {}
for cfg in EXPERIMENTS:
    results[cfg["run_name"]] = T.run_experiment(cfg, data)

## 2. Comparison with `ref_256`

Read the two rows together:
- `ref_256_s43` shows how much the score moves **by chance**, with the same recipe and only the seed changed;
- `cnxs_256` shows how much it moves with the larger backbone.

In [ ]:
RUNS = [cfg["run_name"] for cfg in EXPERIMENTS]
table = T.compare(RUNS, reference=REF)
table.style.format("{:.4f}").background_gradient(subset=["Δ clean", "Δ shifted"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

In [ ]:
# fold by fold (paired, same validation images): Δ = run − ref_256
def fold_f1(run):
    oof = pd.read_csv(T.OUT_DIR / run / "oof.csv")
    out = {}
    for val in ["clean", "shifted"]:
        P = oof[[f"p{c}_{val}" for c in range(C.N_CLASSES)]].to_numpy()
        out[val] = [f1_score(oof.label[oof.fold == k], P[oof.fold == k].argmax(1), average="weighted")
                    if (oof.fold == k).any() else np.nan for k in range(5)]
    return out

ref = fold_f1(REF)
rows = []
for run in RUNS:
    if not (T.OUT_DIR / run / "oof.csv").exists():
        continue
    f = fold_f1(run)
    for val in ["clean", "shifted"]:
        d = np.array(f[val]) - np.array(ref[val])
        rows.append({"run": run, "validation": val, **{f"fold {k}": d[k] for k in range(5)},
                     "mean Δ": np.nanmean(d), "folds better": int((d > 0).sum())})
pd.DataFrame(rows).set_index(["run", "validation"]).style.format(
    {**{f"fold {k}": "{:+.4f}" for k in range(5)}, "mean Δ": "{:+.4f}"}
).background_gradient(subset=["mean Δ"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

## 3. Per class: does capacity help the hard classes?

If capacity is the limit, the gains should appear on **disturbed (0)** and the **spirals (5, 6, 7)**. The easy classes are already at ≈ 0.93–0.95 and have little room left.

In [ ]:
names = [REF] + RUNS
for val in ["clean", "shifted"]:
    t = T.per_class(names, val)
    d = t.sub(t[REF], axis=0).drop(columns=REF)
    print(f"{val.upper()} validation: per-class F1 of {REF}, then Δ")
    display(pd.concat([t[[REF]], d], axis=1).style.format("{:+.3f}").format("{:.3f}", subset=[REF])
            .background_gradient(subset=list(d.columns), cmap="RdYlGn", vmin=-0.05, vmax=0.05))

In [ ]:
# confusion between disturbed and the three spiral classes (clean validation, OOF, rows = true class)
from sklearn.metrics import confusion_matrix
cls = [0, 5, 6, 7]
fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 4.3))
for ax, run in zip(np.atleast_1d(axes), names):
    oof = pd.read_csv(T.OUT_DIR / run / "oof.csv")
    pred = oof[[f"p{c}_clean" for c in range(C.N_CLASSES)]].to_numpy().argmax(1)
    cm = confusion_matrix(oof.label, pred, labels=list(range(C.N_CLASSES)))
    sub = cm[np.ix_(cls, cls)] / cm[cls].sum(1, keepdims=True).clip(min=1)
    ax.imshow(sub, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(cls)):
        for j in range(len(cls)):
            ax.text(j, i, f"{sub[i, j]:.2f}", ha="center", va="center", color="white" if sub[i, j] > 0.5 else "black")
    lab = [f"{c} {C.CLASSES[c]}" for c in cls]
    ax.set_xticks(range(len(cls)), lab, rotation=30, ha="right", fontsize=8); ax.set_yticks(range(len(cls)), lab, fontsize=8)
    ax.set_title(run)
plt.suptitle("Share of each true class (rows) predicted as each class (columns), restricted to 4 classes")
plt.tight_layout(); plt.show()

## 4. Training curves and time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
for name in names:
    f = T.OUT_DIR / name / "history.csv"
    if not f.exists():
        continue
    h = pd.read_csv(f).groupby("epoch").mean(numeric_only=True)
    axes[0].plot(h.index, h.train_loss, label=name)
    axes[1].plot(h.index, h.f1_clean, label=name)
    axes[2].plot(h.index, h.f1_shifted, label=name)
for ax, t in zip(axes, ["train loss", "F1 clean (mean of folds, no TTA)", "F1 shifted (mean of folds, no TTA)"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
axes[1].set_ylim(0.7, 0.9); axes[2].set_ylim(0.7, 0.9)
plt.tight_layout(); plt.show()
print({name: f"{r['minutes']:.1f} min" for name, r in results.items()})

## 5. Decision

| run | F1 clean (Δ) | F1 shifted (Δ) | folds better (clean / shifted) | minutes |
|---|---|---|---|---|
| ref_256 | 0.8510 | 0.8452 | – | 15.9 |
| ref_256_s43 (noise) | ___ | ___ | ___ | ___ |
| cnxs_256 | ___ | ___ | ___ | ___ |

- Seed-to-seed difference of Tiny (the noise): ___
- Does Small beat Tiny by ≳ 0.01 and clearly more than the noise? ___
- Do the gains concentrate on disturbed and the spirals? ___
- **Decision:** ___